# Exercise 5.2 — Evaluating a classification model on training and test data (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-02-classification-metrics.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

Exercise 5.1 evaluated a regression model by the size of its errors. A
classification model predicts a class, diabetes or no diabetes, so an error is
simply a wrong class, and there are two kinds of wrong. In this exercise we fit a
logistic regression on the training rows, turn its probabilities into classes
with a threshold, and measure its performance on the training and the test
rows, first with metrics that depend on the threshold and then with the ROC
curve and the AUC, which do not. Finally we add more predictors and compare.

**Theory:** predicted probabilities and predicted classes; accuracy, precision,
recall, sensitivity and specificity; which class is the event; the distribution
of predicted probabilities; the ROC curve and the area under it (AUC);
underfitting in classification.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_curve, roc_auc_score

The same 750 people and the same split as Exercise 5.1, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv). This time we keep five predictors,
because Part 4 uses them all. Triceps skinfold, insulin and blood pressure are
left out because in this data set a zero in those columns means "not measured".
The outcome is coded 1 for `pos` and 0 for `neg`.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

print(train["diabetes"].value_counts())
print(test["diabetes"].value_counts())

Always look at the number of people in each class first. 220 of the 600
training rows (36.7%) and 53 of the 150 test rows (35.3%) have diabetes.

------------------------------------------------------------------------

### Part 1 — An age-only model, and its predictions

Start with a very simple model: diabetes from age only. `C=np.inf` switches off
scikit-learn's default penalty, so the coefficients match R's.

In [ ]:
age_model = LogisticRegression(C=np.inf)
age_model = age_model.fit(train[["age"]], train["diabetes"])

print("intercept:", round(age_model.intercept_[0], 4))
print("age:      ", round(age_model.coef_[0, 0], 4))

The coefficient of age is positive: older people have higher predicted odds of
diabetes, by a factor of $e^{0.0363} = 1.04$ per year.

A classification model first gives a **probability** for each person. To turn
it into a predicted class we choose a threshold, as in Exercise 4.3; here we use
the usual 0.5. Predict the training rows:

In [ ]:
train_preds = train.copy()
train_preds["prob"] = age_model.predict_proba(train[["age"]])[:, 1]   # probability of 1
train_preds["pred_class"] = (train_preds["prob"] >= 0.5).astype(int)

train_preds.head().round(3)

`prob` is the predicted probability of diabetes (1) and `pred_class` the
predicted class at the threshold 0.5. Now the same for the test rows:

In [ ]:
test_preds = test.copy()
test_preds["prob"] = age_model.predict_proba(test[["age"]])[:, 1]
test_preds["pred_class"] = (test_preds["prob"] >= 0.5).astype(int)

test_preds["pred_class"].value_counts()

At 0.5 the age-only model predicts diabetes for only 20 of the 150 test patients:
a probability of 0.5 or more needs an age of 49 or more.

> **Learning points**
>
> - The same workflow steps as for regression: specify, write the recipe, fit on the training rows, predict both sets.
> - A classifier gives probabilities first; the threshold turns them into classes.

------------------------------------------------------------------------

### Part 2 — Metrics on the training and the test rows

From the confusion table of Exercise 4.3 come several metrics:

- **Accuracy**: the proportion of people classified correctly.
- **Precision**: of the people predicted to have diabetes, the proportion who
  have it, $TP / (TP + FP)$.
- **Recall** or **sensitivity**: of the people who have diabetes, the
  proportion the model finds, $TP / (TP + FN)$.
- **Specificity**: of the people without diabetes, the proportion correctly
  left alone, $TN / (TN + FP)$.

scikit-learn treats 1 as the event (`pos_label=1`). It has no specificity
function, but specificity is the recall of the other class, so we ask for
`recall_score(..., pos_label=0)`.

In [ ]:
truth     = train_preds["diabetes"]
predicted = train_preds["pred_class"]

print("accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("precision:  ", round(precision_score(truth, predicted), 3))
print("recall:     ", round(recall_score(truth, predicted), 3))
print("sensitivity:", round(recall_score(truth, predicted), 3))
print("specificity:", round(recall_score(truth, predicted, pos_label=0), 3))

Recall and sensitivity are the same thing under two names. Now compute the same
metrics on the **test** rows:

> **Stuck?**
>
> The truth is the `diabetes` column of `test_preds`. Specificity is the recall
> of the class 0.

> **Worked answer**

In [ ]:
truth     = test_preds["diabetes"]
predicted = test_preds["pred_class"]

print("accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("precision:  ", round(precision_score(truth, predicted), 3))
print("recall:     ", round(recall_score(truth, predicted), 3))
print("sensitivity:", round(recall_score(truth, predicted), 3))
print("specificity:", round(recall_score(truth, predicted, pos_label=0), 3))

> | | Accuracy | Precision | Recall = sensitivity | Specificity |
> |:-|:-:|:-:|:-:|:-:|
> | Training rows | 0.623 | 0.461 | 0.159 | 0.892 |
> | Test rows | 0.647 | 0.500 | 0.189 | 0.897 |

**Question!** Is the model overfitting? Is it any good? Compare the test accuracy
with a "model" that predicts no diabetes for everybody: it would be right for
the 97 of 150 test patients without diabetes, an accuracy of 0.647.

Training and test metrics are similar, so there is no strong sign of
overfitting. But the age-only model is no more accurate than predicting "no
diabetes" for everyone, and it finds fewer than one in five of the people with
diabetes. It **underfits**: age alone is not enough.

> **Learning points**
>
> - Accuracy, precision, recall (= sensitivity) and specificity all depend on the threshold.
> - Say which class is the event: `event_level = "second"` in yardstick when the event is the second level; `pos_label` in scikit-learn.
> - Compare accuracy with the share of the largest class: a model must beat predicting the same class for everyone.

------------------------------------------------------------------------

### Part 3 — Score distributions and the ROC curve

The metrics above looked at one threshold. To see the model's probabilities
themselves, plot them separately for the people with and without diabetes. A
useful model gives the people with diabetes higher probabilities:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, data, name in [(axes[0], train_preds, "Training rows"), (axes[1], test_preds, "Test rows")]:
    ax.boxplot([data.loc[data["diabetes"] == 0, "prob"], data.loc[data["diabetes"] == 1, "prob"]],
               tick_labels=["no diabetes (0)", "diabetes (1)"])
    ax.set_title(name)
    ax.set_xlabel("True class")
axes[0].set_ylabel("Predicted probability of diabetes")
plt.show()

The boxes overlap a great deal, in both sets: age separates the two groups only
weakly. Every threshold we could draw across this plot would misclassify many
people.

The **ROC curve** shows all thresholds at once. For each threshold it plots the
sensitivity against $1 - \text{specificity}$, the proportion of false alarms;
you computed three of its points by hand in Exercise 4.3. A model that guesses
at random follows the diagonal. The **area under the curve** (AUC) summarises
the curve in one number: 0.5 for guessing, 1 for perfect separation. It is also
the probability that a randomly chosen person with diabetes gets a higher
predicted probability than a randomly chosen person without.

In [ ]:
fpr_age, tpr_age, thresholds_age = roc_curve(test_preds["diabetes"], test_preds["prob"])
auc_age = roc_auc_score(test_preds["diabetes"], test_preds["prob"])

print("test AUC, age only:", round(auc_age, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr_age, tpr_age, color="black")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--")
ax.set_xlabel("1 - specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC curve, age-only model (test rows)")
plt.show()

Now compute the AUC on the **training** rows, and compare:

> **Worked answer**

In [ ]:
auc_age_train = roc_auc_score(train_preds["diabetes"], train_preds["prob"])

print("training AUC, age only:", round(auc_age_train, 3))

> Training AUC 0.671, test AUC 0.668: similar, and both far from 1.

> **Learning points**
>
> - Plot the predicted probabilities by true class: the less the groups overlap, the better the model separates them.
> - The ROC curve shows sensitivity against 1 - specificity for every threshold; the AUC summarises it, 0.5 for guessing and 1 for perfect.
> - AUC does not depend on a threshold, so it compares models before anyone has chosen one.

------------------------------------------------------------------------

### Part 4 — Include more features

Age alone was not enough. Add the other four predictors. The pipeline also
standardises them with `StandardScaler()` (Day 2), so that the coefficients are
on the same scale and can be compared: each is the change in log-odds for one
standard deviation of that predictor.

In [ ]:
predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

rich_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(C=np.inf)),
])
rich_pipe = rich_pipe.fit(train[predictors], train["diabetes"])

pd.Series(rich_pipe["model"].coef_[0], index=predictors).round(3)

Glucose has by far the largest coefficient, 1.02 log-odds per standard
deviation, followed by body mass (0.46). Once glucose is in the model, age adds
little (0.15). (R's `step_normalize()` divides by the standard deviation with
$n - 1$, Python's `StandardScaler()` with $n$; the coefficients differ in the
fourth digit, the predictions do not.)

Predict both sets and compute the metrics:

In [ ]:
train_preds2 = train.copy()
train_preds2["prob"] = rich_pipe.predict_proba(train[predictors])[:, 1]
train_preds2["pred_class"] = (train_preds2["prob"] >= 0.5).astype(int)

test_preds2 = test.copy()
test_preds2["prob"] = rich_pipe.predict_proba(test[predictors])[:, 1]
test_preds2["pred_class"] = (test_preds2["prob"] >= 0.5).astype(int)

for name, data in [("train", train_preds2), ("test", test_preds2)]:
    truth, predicted = data["diabetes"], data["pred_class"]
    print(name,
          "accuracy", round(accuracy_score(truth, predicted), 3),
          "precision", round(precision_score(truth, predicted), 3),
          "sensitivity", round(recall_score(truth, predicted), 3),
          "specificity", round(recall_score(truth, predicted, pos_label=0), 3))

Test accuracy rises from 0.647 to 0.767 and sensitivity from 0.189 to 0.528.
Training and test are still close, so the richer model does not overfit
either. Complete the code for its ROC curve and AUC on the test rows:

> **Worked answer**

In [ ]:
fpr_rich, tpr_rich, thresholds_rich = roc_curve(test_preds2["diabetes"], test_preds2["prob"])
auc_rich = roc_auc_score(test_preds2["diabetes"], test_preds2["prob"])

print("test AUC, five predictors:", round(auc_rich, 3))

> Test AUC 0.830, against 0.668 for age only.

The two ROC curves together. The richer model's curve lies further from the
diagonal at nearly every threshold:

In [ ]:
fpr2, tpr2, thr2 = roc_curve(test_preds2["diabetes"], test_preds2["prob"])

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr_age, tpr_age, linewidth=2, label="age only")
ax.plot(fpr2, tpr2, linewidth=2, label="five predictors")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--")
ax.set_xlabel("1 - specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC curves on the test rows")
ax.legend()
plt.show()

**Question!** The richer model is better on every metric. Could adding even more
predictors ever make a model worse on the test rows? What would you look for?

> **Learning points**
>
> - Adding relevant predictors improved every metric and the AUC, on training and test alike.
> - Standardise the predictors when you want to compare the sizes of their coefficients.
> - More flexible models can still overfit: keep comparing training and test.

## Stretch

> **Stretch — the threshold for the richer model**
>
> Exercise 4.3 chose a threshold by giving a missed case five times the cost of
> an unnecessary test. Repeat that calculation for the five-predictor model on the
> test rows: which threshold is cheapest now, and how much cheaper than the
> age-only model at its best threshold?